# ⏱️ Notebook 5: Active Pulse Time of Arrival (ToA) & TDOA Triangulation
Welcome to the **Acoustic Time of Arrival (ToA) & Direction of Arrival Laboratory** (`v1.3.0`).

This notebook demonstrates active acoustic distance sounding, metric range estimation, and bearing triangulation using a hardware-triggered pulse emitter and dual-microphone receivers on the **PYNQ-Z2 board (`xc7z020clg400-1`)**:

---
### 🏛️ Operating Principles
1. **Synchronous Hardware Emission ($10\,\text{ns}$ Resolution):**
   The FPGA drives Arduino digital pin **`AR2` (Zynq Pin `U13`)** HIGH for a programmed duration (e.g. $10.0\,\text{ms}$) via the `axis_trigger_unit` IP, starting cycle 0 on the exact clock edge the pulse launches.
2. **Piezo Transducer Ignition Delay ($t_{\text{offset}} = 1.8080\,\text{ms}$):**
   Active buzzers take a few acoustic cycles for their internal mechanical oscillator to build resonant pressure waves in air:
   $$t_{\text{flight}} = t_{\text{arrival}} - t_{\text{offset}}$$
   $$r = c(T) \cdot t_{\text{flight}}$$
3. **Calibration-Independent Dual-Microphone TDOA Bearing Triangulation:**
   Subtracting arrival times cancels the buzzer turn-on delay ($t_{\text{offset}}$) to within $< 1.0\,\mu\text{s}$:
   $$\Delta t_{12} = t_{\text{arr, mic2}} - t_{\text{arr, mic1}}$$
   $$\Delta r = c(T) \cdot \Delta t_{12}$$
   $$\theta_{\text{TDOA}} = \arcsin\left(\frac{c(T) \cdot \Delta t_{12}}{d}\right)$$

## 1. Master Star-Ground Circuit Architecture
Connect your setup using **Star Grounding** to prevent buzzer switching current from inducing ground bounce into the microphone preamps:

```
                                  PYNQ-Z2 BOARD
   ┌────────────────────────────────────────────────────────────────────────┐
   │  [Power Header]                   [Analog Header]     [Digital Header] │
   │   • 3.3V ──────────────┐           • A0 ───────────┐   • Pin 2 (AR2) ─┐│
   │   • GND Pin 1 (Clean) ─┼───┐       • A1 ─────────┐ │                  ││
   │   • GND Pin 2 (Noisy) ─┼─┐ │                     │ │                  ││
   └────────────────────────┼─┼─┼─────────────────────┼─┼──────────────────┼┘
                            │ │ │                     │ │                  │
 ═══════════════════════════╪═╪═╪═════════════════════╪═╪══════════════════╪════
   BRANCH 1: SENSITIVE ANALOG│ │                     │ │                  │
   • Mic 1 & 2 VCC ─────────┘ │                     │ │                  │
   • Mic 1 & 2 GND ───────────┘                     │ │                  │
   • Mic 1 OUT ─────────────────────────────────────┼─┘                  │
   • Mic 2 OUT ─────────────────────────────────────┘                    │
 ══════════════════════════════════════════════════════════════════════════╪════
   BRANCH 2: NOISY BUZZER ACTUATOR                                         │
   • External 5V (+) ──► Buzzer (+)                                       │
   • Buzzer (-) ───────► Transistor Collector (Pin 3)                      │
   • AR2 (Digital 2) ──► 1kΩ Resistor ──► Transistor Base (Pin 2) ─────────┘
   • Transistor Emitter (Pin 1) ──► External 5V GND (-) AND PYNQ GND Pin 2
```

In [ ]:
# %% [code] Cell 1: Environment Setup & Hardware Initialization
import json
import time
from pathlib import Path
import numpy as np
import scipy.signal as signal
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pynq import allocate
from pynq_localizer import MicrophoneArrayOverlay, KinematicAnalytics

# 1. Initialize Hardware Overlay (Auto-downloads v1.5.2-rc3)
ol = MicrophoneArrayOverlay()
trig = ol.trigger
dma = ol.axi_dma_0

# 2. System Constants
temperature_c = 20.0
c_sound = KinematicAnalytics.speed_of_sound(temperature_c)
profile_path = Path("profiles/active_buzzer_2610hz.json")

if profile_path.exists():
    with open(profile_path, "r", encoding="utf-8") as f:
        p_data = json.load(f)
    f0 = float(p_data.get("f_res_hz", 2609.73))
    t_offset_cal = float(p_data.get("calibrated_toa_offset_ms", 1.8080))
    d_max = float(p_data.get("max_mic_spacing_aoa_cm", 6.58))
else:
    f0 = 2609.73
    t_offset_cal = 1.8080
    d_max = 6.58

mic_distance_m = 0.05  # 5.0 cm baseline
fs = 500_000.0         # 500 kSPS per channel (M=1 Bypass Mode)

print("=" * 75)
print("✅ HARDWARE INITIALIZATION COMPLETE (v1.5.2-rc3)")
print("=" * 75)
print(f"• Sampling Rate         : {fs/1000:.0f} kSPS per channel (M=1 Bypass Mode)")
print(f"• Carrier Frequency f0  : {f0:.2f} Hz")
print(f"• Speed of Sound c(T)   : {c_sound:.2f} m/s (at {temperature_c}°C)")
print(f"• Calibrated Delay t_off: {t_offset_cal:.4f} ms")
print(f"• Array Baseline d      : {mic_distance_m*100:.1f} cm (Aliasing limit: {d_max:.2f} cm)")
print(f"• Pulse Trigger Pin     : Arduino AR2 (Pin U13)")
print("=" * 75)

In [ ]:
# %% [code] Cell 2: Audible Buzzer 1-Second Check
print("🔊 FIRING 1.0-SECOND TEST PULSE ON AR2...")

# Set 1000 ms pulse (100,000,000 cycles @ 100 MHz)
trig.set_pulse_width_ms(1000.0)
trig.fire_pulse()

time.sleep(0.05)
st = trig.mmio.read(trig.REG_STATUS)
is_active = bool(st & trig.STATUS_PULSE_ACTIVE)

print(f"• Hardware Status (0x04) : {hex(st)}")
print(f"• Is Pin AR2 Actively HIGH: {is_active}")

time.sleep(1.0)
st_done = trig.mmio.read(trig.REG_STATUS)
print(f"• Pulse Completed (0x04) : {hex(st_done)} (Active = {bool(st_done & trig.STATUS_PULSE_ACTIVE)})")

if is_active:
    print("\n👉 Did you hear the loud 1-second beep from the buzzer?")
else:
    print("\n❌ Hardware pulse did not trigger. Check connections.")

In [ ]:
# %% [code] Cell 3: Single-Shot Dual-Channel Acoustic Sounding (10 ms Pulse)
input("👉 Place buzzer at target location (e.g. Mic 2 @ 17 cm, Mic 1 @ 61.5 cm) and press [Enter]...")

res = ol.capture_pulsed_toa_frame(
    pulse_width_ms=10.0,
    packet_samples=16384,
    f_target=f0,
    mic_distance_m=mic_distance_m,
    profile=profile_path,
    temperature_c=temperature_c
)

print("=" * 80)
print("📊 DUAL-CHANNEL ACOUSTIC ToA & TDOA RANGING REPORT")
print("=" * 80)
print(f"• Operational Status         : ✅ {res['status']}")
print(f"• Pre-Acoustic Spike (t<0.25): Mic 1 = {np.max(res['env1'][:125]):5.1f} mV | Mic 2 = {np.max(res['env2'][:125]):5.1f} mV (Star Ground Clean)")
print("-" * 80)
print(f"• Mic 2 (A1, Near) Arrival   : {res['t2_raw_ms']:6.3f} ms ──► Inverted Dist = {res['r2_cm']:5.1f} cm")
print(f"• Mic 1 (A0, Far) Arrival    : {res['t1_raw_ms']:6.3f} ms ──► Inverted Dist = {res['r1_cm']:5.1f} cm")
print("-" * 80)
print(f"• Physical Path Delta (Δr)   : {res['delta_r_cm']:+5.1f} cm (Δt12 = {res['delta_t_ms']*1000.0:+6.1f} µs)")
if np.isfinite(res['theta_tdoa_deg']):
    print(f"• Incident Bearing Angle (θ) : {res['theta_tdoa_deg']:+5.1f}°")
print(f"• Peak Burst Amplitudes      : Mic 1 = {res['amp_a0_v']*1000.0:5.1f} mV | Mic 2 = {res['amp_a1_v']*1000.0:5.1f} mV")
print("=" * 80)

In [ ]:
# %% [code] Cell 4: Interactive Waveform Diagnostic Plot
t_ms = res["t_ms"]
mask_zoom = t_ms <= 8.0

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        f"<b>Mic 1 (A0, Far): Arrival = {res['t1_raw_ms']:.3f} ms ──► Inverted Dist = {res['r1_cm']:.1f} cm</b>",
        f"<b>Mic 2 (A1, Near): Arrival = {res['t2_raw_ms']:.3f} ms ──► Inverted Dist = {res['r2_cm']:.1f} cm</b>"
    )
)

# Mic 1
fig.add_scatter(x=t_ms[mask_zoom], y=res["v1_bp"][mask_zoom], mode="lines", line=dict(color="#00FFCC", width=1.5), name="Mic 1 Sine", row=1, col=1)
fig.add_scatter(x=t_ms[mask_zoom], y=res["env1"][mask_zoom], mode="lines", line=dict(color="#FFA500", width=1.8), name="Mic 1 Env", row=1, col=1)
if np.isfinite(res["t1_raw_ms"]):
    fig.add_vline(x=res["t1_raw_ms"], line=dict(color="red", dash="dash", width=2), annotation_text=f"Arrival: {res['t1_raw_ms']:.3f} ms", row=1, col=1)

# Mic 2
fig.add_scatter(x=t_ms[mask_zoom], y=res["v2_bp"][mask_zoom], mode="lines", line=dict(color="#FF007F", width=1.5), name="Mic 2 Sine", row=2, col=1)
fig.add_scatter(x=t_ms[mask_zoom], y=res["env2"][mask_zoom], mode="lines", line=dict(color="#FFA500", width=1.8), name="Mic 2 Env", row=2, col=1)
if np.isfinite(res["t2_raw_ms"]):
    fig.add_vline(x=res["t2_raw_ms"], line=dict(color="red", dash="dash", width=2), annotation_text=f"Arrival: {res['t2_raw_ms']:.3f} ms", row=2, col=1)

fig.update_layout(
    template="plotly_dark", height=550,
    title=f"<b>Acoustic Wavefront Delay Verification (Measured Δr = {res['delta_r_cm']:+.1f} cm)</b>",
    xaxis2=dict(title="Time from Pulse Launch (ms)", range=[0, 8.0]),
    yaxis=dict(title="Mic 1 (mV)", range=[-250, 250]),
    yaxis2=dict(title="Mic 2 (mV)", range=[-250, 250])
)
fig.show()

In [ ]:
# %% [code] Cell 5: Multi-Distance Radial Linearity Benchmark
test_distances_cm = [25.0, 50.0, 75.0, 100.0]
measured_dists = []

print("=" * 75)
print("📏 MULTI-DISTANCE RADIAL LINEARITY BENCHMARK")
print("=" * 75)

for r_tgt in test_distances_cm:
    input(f"👉 Move buzzer to {r_tgt:.0f} cm along ruler from Mic 1 and press [Enter]...")
    
    burst_runs = []
    for _ in range(5):
        run = ol.capture_pulsed_toa_frame(
            pulse_width_ms=10.0,
            packet_samples=16384,
            f_target=f0,
            profile=profile_path,
            temperature_c=temperature_c
        )
        if run["status"] == "ACTIVE_VALID":
            burst_runs.append(run["r1_cm"])
        time.sleep(0.10)
        
    m_dist = float(np.mean(burst_runs)) if burst_runs else np.nan
    measured_dists.append(m_dist)
    err_cm = abs(m_dist - r_tgt)
    print(f"   • Target: {r_tgt:5.1f} cm ──► Measured: {m_dist:5.1f} cm (Error = {err_cm:4.1f} cm)")

# Linearity Plot
valid_mask = np.isfinite(measured_dists)
x_vals = np.array(test_distances_cm)[valid_mask]
y_vals = np.array(measured_dists)[valid_mask]

fig_dist = go.Figure()
fig_dist.add_scatter(x=[0, 115], y=[0, 115], mode="lines", line=dict(color="gray", dash="dash"), name="Ideal (y = x)")
fig_dist.add_scatter(x=x_vals, y=y_vals, mode="lines+markers", marker=dict(size=9, color="#00FFCC"), line=dict(color="#00FFCC", width=2), name="Measured Distance")

fig_dist.update_layout(template="plotly_dark", height=450, title="<b>Acoustic ToA Distance Sounding Linearity</b>")
fig_dist.update_xaxes(title="True Physical Distance (cm)", range=[0, 120])
fig_dist.update_yaxes(title="Estimated ToA Distance (cm)", range=[0, 120])
fig_dist.show()

In [ ]:
# %% [code] Cell 6: 10-Shot Repeatability & Jitter Benchmark
n_shots = 10
shot_r1 = []
shot_r2 = []
shot_dr = []

print("=" * 85)
print(f"🚀 FIRING {n_shots} CONSECUTIVE PULSES (120 ms acoustic rest between shots)...")
print("=" * 85)
print(f"{'Shot #':<8} | {'Mic 1 Dist':<15} | {'Mic 2 Dist':<15} | {'Δr (cm)':<12} | {'Status'}")
print("-" * 85)

for p in range(n_shots):
    frame = ol.capture_pulsed_toa_frame(
        pulse_width_ms=10.0,
        packet_samples=16384,
        f_target=f0,
        profile=profile_path,
        temperature_c=temperature_c
    )
    if frame["status"] == "ACTIVE_VALID":
        shot_r1.append(frame["r1_cm"])
        shot_r2.append(frame["r2_cm"])
        shot_dr.append(frame["delta_r_cm"])
        print(f"#{p+1:<7} | {frame['r1_cm']:6.2f} cm        | {frame['r2_cm']:6.2f} cm        | {frame['delta_r_cm']:+6.2f} cm   | ✅ VALID")
    else:
        print(f"#{p+1:<7} | {'---':<15} | {'---':<15} | {'---':<12} | ⚠️ {frame['status']}")
    time.sleep(0.12)

print("=" * 85)
if len(shot_r1) >= 3:
    jitter_r1_mm = float(np.std(shot_r1)) * 10.0
    jitter_r2_mm = float(np.std(shot_r2)) * 10.0
    jitter_dr_mm = float(np.std(shot_dr)) * 10.0
    print("📊 REPEATABILITY JITTER ANALYSIS:")
    print(f"  • Mic 1 Distance (r1) : Mean = {np.mean(shot_r1):.2f} cm | Jitter (σ) = ±{jitter_r1_mm:.2f} mm")
    print(f"  • Mic 2 Distance (r2) : Mean = {np.mean(shot_r2):.2f} cm | Jitter (σ) = ±{jitter_r2_mm:.2f} mm")
    print(f"  • Path Delta (Δr)     : Mean = {np.mean(shot_dr):+6.2f} cm | Jitter (σ) = ±{jitter_dr_mm:.2f} mm")
print("=" * 85)

In [ ]:
# %% [code] Cell 7: Clean Teardown & FPGA Memory Release
ol.close()
print("🔒 FPGA hardware resources cleanly released.")